# S&P 500 Screener v2

Czysta wersja. Parametry edytujesz na górze. Kliknij Runtime > Run all.

In [ ]:
!pip install yfinance pandas tqdm openpyxl requests -q

import yfinance as yf
import pandas as pd
from tqdm import tqdm
import requests
from io import StringIO
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ==================== PARAMETRY DO EDYCJI ====================
MIN_MARKET_CAP_BILLION = 10
MAX_PE = 35
MIN_ROE_PCT = 12
MIN_PROFIT_MARGIN_PCT = 10
MIN_REVENUE_GROWTH_PCT = 8
MAX_DEBT_TO_EQUITY = 1.2
MIN_RSI = 0
MAX_RSI = 65
MIN_VOLUME_INCREASE_PCT = 30

def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    resp = requests.get(url, headers=headers, timeout=15)
    tables = pd.read_html(StringIO(resp.text), header=0)
    return tables[0]['Symbol'].dropna().tolist()

def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def analyze_ticker(ticker):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        hist = stock.history(period='6mo', auto_adjust=True)
        if len(hist) < 50: return None
        market_cap = info.get('marketCap', 0) or 0
        pe = info.get('trailingPE') or info.get('forwardPE') or 999
        roe = (info.get('returnOnEquity') or 0) * 100
        profit_margin = (info.get('profitMargins') or 0) * 100
        rev_growth = (info.get('revenueGrowth') or 0) * 100
        debt_equity = info.get('debtToEquity') or 999
        hist['vol_ma'] = hist['Volume'].rolling(20).mean()
        vol_increase = ((hist['Volume'].iloc[-1] / hist['vol_ma'].iloc[-1]) - 1) * 100
        rsi = calculate_rsi(hist['Close']).iloc[-1]
        # Prosty trend (cena powyżej średniej + wolumen)
        sma20 = hist['Close'].rolling(20).mean().iloc[-1]
        above_sma = hist['Close'].iloc[-1] > sma20
        # Scoring
        score = 0
        if market_cap > MIN_MARKET_CAP_BILLION * 1_000_000_000: score += 20
        if 0 < pe < MAX_PE: score += 15
        if roe > MIN_ROE_PCT: score += 15
        if profit_margin > MIN_PROFIT_MARGIN_PCT: score += 10
        if rev_growth > MIN_REVENUE_GROWTH_PCT: score += 10
        if debt_equity < MAX_DEBT_TO_EQUITY: score += 10
        if vol_increase > MIN_VOLUME_INCREASE_PCT: score += 15
        if above_sma: score += 10
        if MAX_RSI > rsi > MIN_RSI: score += 10
        grade = 'A' if score >= 80 else 'B' if score >= 65 else 'C' if score >= 50 else 'D' if score >= 35 else 'F'
        return {
            'Ticker': ticker,
            'Name': info.get('shortName', ticker),
            'MarketCap_B': round(market_cap / 1e9, 1),
            'PE': round(pe, 1) if pe < 999 else None,
            'ROE': round(roe, 1),
            'ProfitMargin': round(profit_margin, 1),
            'RevGrowth': round(rev_growth, 1),
            'Score': round(score, 1),
            'Grade': grade
        }
    except:
        return None

print('Pobieram listę S&P 500...')
tickers = get_sp500_tickers()
print(f'Znaleziono {len(tickers)} spółek. Analizuję...')
results = []
for t in tqdm(tickers, desc='Scanning'):
    res = analyze_ticker(t)
    if res: results.append(res)
df = pd.DataFrame(results).sort_values('Score', ascending=False)
print('
=== TOP 15 ===')
print(df.head(15).to_string(index=False))
df.to_excel('SP500_Screener_Results.xlsx', index=False)
print('
Zapisano do SP500_Screener_Results.xlsx')